# 08c -- KeepTradeCut Rankings

**Purpose**: Scrape KeepTradeCut dynasty rookie rankings and ingest into
`fact_rookie_rankings`. Self-contained.

**Sources**: Both value sets are extracted from a **single HTTP fetch** (both embedded
in `var playersArray` JSON in the page HTML — no auth required).

| Format scraped | `values_key` | notes |
|---|---|---|
| 1QB | `oneQBValues` | base |
| Superflex | `superflexValues` | base |
| SF TE++ | `superflexValues.tepp` | nested sub-key |

**`grade`**: KTC trade value (0–10 000 crowdsourced scale — not a 0–100 scouting grade).

**Aggregation**: All three formats are averaged per player into a single `KTC_Consensus`
row — `global_rank`, `positional_rank`, and `grade` (trade value) are each the mean across
the three format variants. One HTTP fetch yields all three (sub-keys nested in the same JSON).

**`rank_date`**: set to `TODAY` — KTC updates continuously, no fixed publish date.

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended), `data/review_fuzzy_matches.csv` (if needed)

## 1 -- Setup & Shared Helpers

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import requests
import re
import json
import time
from pathlib import Path
from datetime import date, datetime
from dataclasses import dataclass, field
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from thefuzz import fuzz, process

# ---- Shared helpers + config from etl_helpers -------------------------------
import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (
    LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS, DEFAULT_HEADERS,
    _make_session, _parse_rank_date, clean_player_name, generate_player_key,
    add_players_from_source, ingest_ranking_source, append_review,
)


In [2]:
# clean_player_name / generate_player_key / _parse_rank_date imported from etl_helpers

Helpers OK


## 2 -- add_players_from_source

In [ ]:
# add_players_from_source imported from etl_helpers (canonical, alias-aware)

## 3 -- ingest_ranking_source

In [ ]:
# ingest_ranking_source imported from etl_helpers

## 4 -- KeepTradeCutScraper

`var playersArray` JSON is embedded directly in the HTML — no API key, no auth.
A single `requests.get()` returns all 57 rookies with both `superflexValues` and
`oneQBValues`. The scraper caches the response internally so both sources share one fetch.

In [5]:
class KeepTradeCutScraper:
    # Parses var playersArray JSON embedded in KTC dynasty rookie rankings page.
    # Both superflexValues and oneQBValues are present in every response, so a
    # single HTTP fetch yields two source rows per player.
    #
    # grade field stores the raw KTC trade value (0-10000 crowdsourced scale).
    # rank_date = TODAY because KTC rankings update continuously.

    _URL = "https://keeptradecut.com/dynasty-rankings/rookie-rankings"
    _PLAYERS_PAT = re.compile(r"var playersArray = (\[.*?\]);", re.DOTALL)

    _HEADERS = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }

    SOURCES = {
        "KTC_1QB": {
            "values_key":     "oneQBValues",
            "site_reference": "Dynasty Rookie Rankings - 1QB",
            "source_site":    "KeepTradeCut",
        },
        "KTC_Superflex": {
            "values_key":     "superflexValues",
            "site_reference": "Dynasty Rookie Rankings - Superflex",
            "source_site":    "KeepTradeCut",
        },
        "KTC_TEpp": {
            "values_key":     "superflexValues",
            "sub_key":        "tepp",          # nested: superflexValues.tepp
            "site_reference": "Dynasty Rookie Rankings - SF TE++",
            "source_site":    "KeepTradeCut",
        },
    }

    def __init__(self, timeout: int = 30, delay: float = 2.0):
        self.timeout  = timeout
        self.delay    = delay
        self.session  = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)
        self._cached_players: list | None = None  # reuse across scrape() calls

    def _fetch_players(self) -> list:
        # Fetch page once; cache for subsequent calls in the same session.
        if self._cached_players is not None:
            return self._cached_players
        time.sleep(self.delay)
        resp = self.session.get(self._URL, timeout=self.timeout)
        resp.raise_for_status()
        m = self._PLAYERS_PAT.search(resp.text)
        if not m:
            raise ValueError("var playersArray not found in KTC page HTML")
        self._cached_players = json.loads(m.group(1))
        return self._cached_players

    def scrape(self, source_key: str) -> tuple[pd.DataFrame, str | None]:
        # Returns (df, rank_date). rank_date = TODAY — KTC updates continuously.
        cfg     = self.SOURCES[source_key]
        vkey    = cfg["values_key"]
        sub_key = cfg.get("sub_key")      # optional nested key (e.g., "tepp")
        players = self._fetch_players()

        rows = []
        for i, p in enumerate(players):
            vals = p.get(vkey, {})
            if sub_key:
                vals = vals.get(sub_key, {})  # drill into nested structure
            if not vals:
                continue
            rows.append({
                "player_name":     p["playerName"],
                "position_raw":    p["position"],
                "school_raw":      p.get("college", ""),
                "global_rank":     vals.get("rookieRank"),
                "positional_rank": vals.get("rookiePositionalRank"),
                "grade":           float(vals["value"]) if vals.get("value") is not None else None,
                "ktc_value":       vals.get("value"),
                "ktc_tier":        vals.get("rookieTier"),
                "trend_30d":       vals.get("overallTrend"),
                "nfl_team":        p.get("team", ""),
                "ktc_player_id":   p.get("playerID"),
            })

        if not rows:
            raise ValueError(f"No player rows parsed for {source_key}")

        df = pd.DataFrame(rows)
        rank_date = TODAY  # KTC is continuously updated; use capture date as rank_date
        print(f"Scraped {source_key}: {len(df)} players | {cfg['site_reference']} | rank_date={rank_date}")
        return df, rank_date

## 5 -- Run KeepTradeCut Ingestion

In [6]:
SOURCES = {"KTC_Consensus": {"source_site": "KeepTradeCut"}}  # alias for validation cell

KTC_SCRAPER = KeepTradeCutScraper(timeout=30, delay=2.0)
PHASE       = "post_draft"
_failed     = []
_all_reviews= []
_scraped    = {}  # source_key -> (df, rank_date)

# ── Step 1: Scrape 3 formats — single HTTP fetch via internal cache ────────────
print("Scraping KTC formats (1 fetch, 3 sub-keys)...")
for source_key in KeepTradeCutScraper.SOURCES:
    cfg = KeepTradeCutScraper.SOURCES[source_key]
    try:
        rankings_df, rank_date = KTC_SCRAPER.scrape(source_key)
        _scraped[source_key] = (rankings_df, rank_date)
    except Exception as e:
        print(f"  ERROR {source_key}: {e}")
        _failed.append(source_key)

if not _scraped:
    print("No sources scraped — aborting.")
else:
    # ── Step 2: add_players_from_source once (players identical across formats) ──
    _first_key = next(iter(_scraped))
    _first_df, _ = _scraped[_first_key]
    try:
        _, review = add_players_from_source(
            _first_df, source_name="KTC", draft_year=CFG.draft_year
        )
        if not review.empty:
            print(f"  -> {len(review)} players flagged for review")
            _all_reviews.append(review)
    except Exception as e:
        print(f"  ERROR add_players: {e}")

    # ── Step 3: Aggregate 3 formats -> one consensus row per player ────────────
    # global_rank, positional_rank, and grade (trade value) are each averaged
    # across 1QB / Superflex / TE++.
    _all_dfs = []
    for source_key, (df, rank_date) in _scraped.items():
        _d = df.copy()
        _d["_format"]     = source_key
        _d["_rank_date"]  = rank_date
        _all_dfs.append(_d)

    _combined = pd.concat(_all_dfs, ignore_index=True)
    _combined["_name_clean"] = _combined["player_name"].apply(clean_player_name)

    agg_ktc = (
        _combined.groupby("_name_clean", dropna=False)
        .agg(
            player_name     = ("player_name",     "first"),
            position_raw    = ("position_raw",    "first"),
            school_raw      = ("school_raw",      "first"),
            global_rank     = ("global_rank",     "mean"),
            positional_rank = ("positional_rank", "mean"),
            grade           = ("grade",           "mean"),   # avg trade value (0-10000 scale)
            rank_date       = ("_rank_date",      "max"),
            _formats        = ("_format",         lambda x: "+".join(sorted(set(x)))),
        )
        .reset_index(drop=True)
    )
    agg_ktc["global_rank"]     = agg_ktc["global_rank"].round().astype("Int64")
    agg_ktc["positional_rank"] = agg_ktc["positional_rank"].round().astype("Int64")

    n_formats = len(_scraped)
    print(f"\n{n_formats} formats x {len(_combined)//n_formats} players "
          f"-> {len(agg_ktc)} KTC_Consensus rows")
    print("\nTop 10 consensus:")
    print(agg_ktc.sort_values("global_rank")[
        ["player_name","position_raw","global_rank","positional_rank","grade"]
    ].head(10).to_string(index=False))

    # ── Step 4: Ingest KTC_Consensus ──────────────────────────────────────────
    try:
        ingest_ranking_source(
            agg_ktc,
            source_name="KTC_Consensus",
            source_site="KeepTradeCut",
            phase=PHASE,
            draft_year=CFG.draft_year,
            rank_date=agg_ktc["rank_date"].max(),
        )
    except Exception as e:
        print(f"  ERROR ingesting KTC_Consensus: {e}")
        _failed.append("KTC_Consensus")

    # ── Review file (append-safe across notebooks) ─────────────────────────────
    if _all_reviews:
        new_review = pd.concat(_all_reviews, ignore_index=True)
        rp = DATA / "review" / "review_fuzzy_matches.csv"
        if rp.exists():
            existing = pd.read_csv(rp)
            combined = pd.concat([existing, new_review], ignore_index=True)
            # keep="first" preserves already-filled action values from prior review sessions
            combined.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
            n_added = len(combined) - len(existing)
            combined.to_csv(rp, index=False)
            print("\n" + "=" * 60)
            print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined)} total) -> {rp}")
        else:
            new_review.to_csv(rp, index=False)
            print("\n" + "=" * 60)
            print(f"Review file: {rp}  ({len(new_review)} rows)")
        print("Fill \"action\" column (\"match\" or \"new\"), run 09_apply_fuzzy_review, then re-run.")

    print("\n" + "=" * 60)
    if _failed:
        print(f"Failed: {_failed}")
    else:
        print("KTC_Consensus ingested.")

Scraping KTC formats (1 fetch, 3 sub-keys)...
Scraped KTC_1QB: 57 players | Dynasty Rookie Rankings - 1QB | rank_date=2026-05-16
Scraped KTC_Superflex: 57 players | Dynasty Rookie Rankings - Superflex | rank_date=2026-05-16
Scraped KTC_TEpp: 57 players | Dynasty Rookie Rankings - SF TE++ | rank_date=2026-05-16
Source: KTC
  Already in dim_rookie_prospect : 54
  Auto-matched (>=90)       : 0
  New prospects added             : 0
  Needs manual review             : 3
  -> 3 players flagged for review

3 formats x 57 players -> 57 KTC_Consensus rows

Top 10 consensus:
     player_name position_raw  global_rank  positional_rank       grade
  Jeremiyah Love           RB            1                1 7775.333333
    Carnell Tate           WR            2                1 6060.666667
    Jordyn Tyson           WR            4                2 5523.666667
Fernando Mendoza           QB            5                1 5336.666667
     Makai Lemon           WR            5                3 5296.666

## 6 -- Validation

In [7]:
# Mini-validation: rows written by this source
import pandas as pd
from pathlib import Path
fr = pd.read_parquet(Path(CFG.data_dir) / "fact_rookie_rankings.parquet")
src_keys = list(SOURCES.keys())
sub = fr[fr["source_name"].isin(src_keys)]
print(f"Rows for this source: {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().to_string())
print()
print(f"fact_rookie_rankings total: {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key","source_name","phase","draft_year"]).sum()
print(f"Composite key duplicates: {dupes} (expected 0)")

Rows for this source: 54
source_name    phase     
KTC_Consensus  post_draft    54

fact_rookie_rankings total: 363 rows
Composite key duplicates: 0 (expected 0)
